In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

TEST_DAYS = 60
AVG_UNIT_PROFIT = 8.50


def load_and_check(filename):
    df = pd.read_csv(filename, parse_dates=["date"])
    df = df.sort_values("date").reset_index(drop=True)

    expected_days = (df["date"].max() - df["date"].min()).days + 1
    actual_days = len(df)
    gaps = expected_days - actual_days

    print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
    print("Expected days:", expected_days, "actual rows:", actual_days, "missing:", gaps)

    df = df.set_index("date").asfreq("D")
    df["sales"] = df["sales"].interpolate()

    return df


def decompose_and_plot(df, filename):
    decomposition = seasonal_decompose(df["sales"], model="additive", period=7)

    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

    axes[0].plot(df.index, df["sales"])
    axes[0].set_title("Original Daily Sales")

    axes[1].plot(df.index, decomposition.trend)
    axes[1].set_title("Trend Component")

    axes[2].plot(df.index, decomposition.seasonal)
    axes[2].set_title("Weekly Seasonal Component")
    axes[2].set_xlim(df.index[0], df.index[60])

    axes[3].plot(df.index, decomposition.resid)
    axes[3].set_title("Residual (Noise)")

    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

    return decomposition


def time_based_split(df, test_days):
    train = df.iloc[:-test_days]
    test = df.iloc[-test_days:]
    print("Train:", len(train), "days, Test:", len(test), "days")
    return train, test


def seasonal_naive_forecast(train, test_len):
    last_week = train["sales"].values[-7:]
    reps = int(np.ceil(test_len / 7))
    forecast = np.tile(last_week, reps)[:test_len]
    return forecast


def moving_average_forecast(train, test_len, window=14):
    avg = train["sales"].values[-window:].mean()
    return np.full(test_len, avg)


def holt_winters_forecast(train, test_len):
    model = ExponentialSmoothing(
        train["sales"], trend="add", seasonal="add", seasonal_periods=7
    ).fit()
    return model.forecast(test_len).values


def build_features(df, start_index):
    X = pd.DataFrame(index=df.index)
    X["day_index"] = range(start_index, start_index + len(df))

    for i in range(7):
        X["dow_" + str(i)] = (df.index.dayofweek == i).astype(int)

    return X


def linear_regression_forecast(train, test):
    X_train = build_features(train, 0)
    X_test = build_features(test, len(train))

    model = LinearRegression()
    model.fit(X_train, train["sales"])
    return model.predict(X_test)


def evaluate(actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    mse = mean_squared_error(actual, predicted)
    rmse = mse ** 0.5
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100

    result = {"mae": round(mae, 2), "rmse": round(rmse, 2), "mape": round(mape, 2)}
    return result


def run_all_forecasts(train, test):
    actual = test["sales"].values
    test_len = len(test)

    forecasts = {
        "Seasonal Naive (baseline)": seasonal_naive_forecast(train, test_len),
        "Moving Average (14-day)": moving_average_forecast(train, test_len),
        "Holt-Winters Exponential Smoothing": holt_winters_forecast(train, test_len),
        "Linear Regression (trend + day-of-week)": linear_regression_forecast(train, test)
    }

    results = {}
    for name in forecasts:
        forecast = forecasts[name]
        metrics = evaluate(actual, forecast)
        metrics["forecast"] = forecast
        results[name] = metrics
        print(name, "-> MAE:", metrics["mae"], "RMSE:", metrics["rmse"], "MAPE:", metrics["mape"])

    return results


def plot_forecast_comparison(test, results, filename):
    plt.figure(figsize=(13, 6))
    plt.plot(test.index, test["sales"], label="Actual", color="black", linewidth=2)

    for name in results:
        plt.plot(test.index, results[name]["forecast"], label=name, alpha=0.7, linestyle="--")

    plt.title("Forecast Comparison on Test Period")
    plt.xlabel("Date")
    plt.ylabel("Sales")
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()


def find_best_model(results):
    best_name = None
    best_mae = None

    for name in results:
        if best_mae is None or results[name]["mae"] < best_mae:
            best_mae = results[name]["mae"]
            best_name = name

    return best_name


def write_report(df, decomposition, results, filename):
    best_model_name = find_best_model(results)
    best = results[best_model_name]
    baseline_mae = results["Seasonal Naive (baseline)"]["mae"]

    weekly_pattern = decomposition.seasonal[:7]
    peak_day_name = weekly_pattern.idxmax().strftime("%A")

    daily_cost = best["mae"] * AVG_UNIT_PROFIT
    annual_cost = daily_cost * 365
    daily_savings = (baseline_mae - best["mae"]) * AVG_UNIT_PROFIT
    annual_savings = daily_savings * 365

    file = open(filename, "w")

    file.write("CAPSTONE PROJECT 3: SALES / DEMAND FORECASTING\n")
    file.write("=" * 50 + "\n\n")

    file.write("1. TIME-BASED DATA ANALYSIS\n")
    file.write("-" * 40 + "\n")
    file.write("Date range: " + str(df.index.min().date()) + " to " + str(df.index.max().date()))
    file.write(" (" + str(len(df)) + " days, no gaps)\n\n")

    file.write("2. TREND AND SEASONALITY\n")
    file.write("-" * 40 + "\n")
    trend_start = decomposition.trend.dropna().iloc[0]
    trend_end = decomposition.trend.dropna().iloc[-1]
    file.write("Trend: sales grew from about " + str(round(trend_start)) + "/day to about ")
    file.write(str(round(trend_end)) + "/day over the period.\n")
    file.write(peak_day_name + " is consistently the strongest sales day.\n")
    file.write("A seasonal demand bump also appears in the Oct-Nov window.\n\n")

    file.write("3. FORECASTING MODEL COMPARISON\n")
    file.write("-" * 40 + "\n")
    file.write("Test period: last " + str(TEST_DAYS) + " days (time-ordered split, not random).\n\n")

    for name in results:
        r = results[name]
        file.write(name + ": MAE=" + str(r["mae"]) + ", RMSE=" + str(r["rmse"]))
        file.write(", MAPE=" + str(r["mape"]) + "%\n")

    file.write("\nBest model: " + best_model_name + " (MAE = " + str(best["mae"]) + ")\n")

    if best["mae"] < baseline_mae:
        improvement = round((1 - best["mae"] / baseline_mae) * 100, 1)
        file.write("This beats the baseline by " + str(improvement) + "%, a real improvement.\n")
    else:
        file.write("No model clearly beat the baseline. The simple baseline is fine to use here.\n")

    file.write("\n4. BUSINESS IMPACT ANALYSIS\n")
    file.write("-" * 40 + "\n")
    file.write("Average daily forecast error (best model): " + str(best["mae"]) + " units.\n")
    file.write("At $" + str(AVG_UNIT_PROFIT) + " profit per unit, that is roughly $")
    file.write(str(round(daily_cost)) + " per day in over or under stocking risk.\n")
    file.write("Annualized, about $" + str(round(annual_cost)) + " per year in residual inventory cost")
    file.write(" that forecasting alone cannot remove.\n")
    file.write("Value of switching from the baseline to the best model: about $")
    file.write(str(round(annual_savings)) + " per year saved.\n")

    file.write("\nRECOMMENDATIONS:\n")
    file.write("- Plan extra staff and stock for " + peak_day_name + "s, it is a structural pattern.\n")
    file.write("- Build in extra stock ahead of the Oct-Nov seasonal window.\n")
    file.write("- Use " + best_model_name + " for forecasting, but keep the baseline as a check.\n")
    file.write("  If the model error goes above the baseline error, the pattern has likely shifted.\n")

    file.close()


def main():

    df = load_and_check("daily_sales_raw.csv")
    decomposition = decompose_and_plot(df, "trend_seasonality_charts.png")

    train, test = time_based_split(df, TEST_DAYS)
    results = run_all_forecasts(train, test)

    plot_forecast_comparison(test, results, "forecast_comparison_chart.png")
    write_report(df, decomposition, results, "forecasting_capstone_report.txt")

    print("\nCapstone Project 3 done!")
    print("Files saved: trend_seasonality_charts.png, forecast_comparison_chart.png, forecasting_capstone_report.txt")


if __name__ == "__main__":
    main()

Date range: 2024-01-01 to 2025-12-30
Expected days: 730 actual rows: 730 missing: 0
Train: 670 days, Test: 60 days
Seasonal Naive (baseline) -> MAE: 136.95 RMSE: 158.99 MAPE: 13.87
Moving Average (14-day) -> MAE: 141.97 RMSE: 167.46 MAPE: 14.47
Holt-Winters Exponential Smoothing -> MAE: 158.32 RMSE: 183.33 MAPE: 16.07
Linear Regression (trend + day-of-week) -> MAE: 86.19 RMSE: 113.78 MAPE: 7.88

Capstone Project 3 done!
Files saved: trend_seasonality_charts.png, forecast_comparison_chart.png, forecasting_capstone_report.txt
